# Tier 3 Solutions Set 3: Advanced Methods

Solutions for Assignment Set 3: Advanced Methods.

In [ ]:
import numpy as np
import math
from math import log2
import random
from collections import defaultdict

## Problem 1: Enzyme Kinetics — Michaelis-Menten (1 star)

In [ ]:
def michaelis_menten_fit(data: list[tuple[float, float]]) -> dict:
    """
    Estimate Km and Vmax from (S, V) data via Lineweaver-Burk linear regression.

    Args:
        data: List of (substrate_concentration_S, reaction_rate_V) tuples.
              All values must be positive.

    Returns:
        Dict with keys 'Vmax' and 'Km', each rounded to 4 decimal places.
    """
    # Transform to double-reciprocal (1/S, 1/V)
    transformed = [(1.0/s, 1.0/v) for s, v in data]
    n = len(transformed)

    # Least-squares linear regression: 1/V = m*(1/S) + b
    sum_x  = sum(x for x, y in transformed)
    sum_y  = sum(y for x, y in transformed)
    sum_xy = sum(x*y for x, y in transformed)
    sum_x2 = sum(x*x for x, y in transformed)

    m = (n*sum_xy - sum_x*sum_y) / (n*sum_x2 - sum_x**2)
    b = (sum_y - m*sum_x) / n

    # Recover kinetic parameters
    vmax = 1.0 / b
    km   = m * vmax

    return {"Vmax": round(vmax, 4), "Km": round(km, 4)}


# Test — true Vmax=10, Km=2
test_data = [
    (0.5,  1.25),
    (1.0,  3.33),
    (2.0,  5.00),
    (4.0,  6.67),
    (8.0,  8.00),
    (16.0, 8.89),
]
result = michaelis_menten_fit(test_data)
print(f"Vmax = {result['Vmax']}, Km = {result['Km']}")
# Expected: Vmax ≈ 10.0, Km ≈ 2.0

## Problem 2: Hardy-Weinberg Equilibrium (1 star)

In [ ]:
def hardy_weinberg_test(AA: int, Aa: int, aa: int) -> dict:
    """
    Test a population of genotype counts for Hardy-Weinberg Equilibrium.

    Args:
        AA: Count of homozygous dominant individuals.
        Aa: Count of heterozygous individuals.
        aa: Count of homozygous recessive individuals.

    Returns:
        Dict with keys 'p_allele', 'q_allele', 'chi2', 'in_hwe'.
        Floats rounded to 4 decimal places.
    """
    N = AA + Aa + aa

    # Allele frequencies
    p = (2*AA + Aa) / (2*N)
    q = 1.0 - p

    # Expected genotype counts under HWE
    AA_exp = p**2 * N
    Aa_exp = 2*p*q * N
    aa_exp = q**2 * N

    # Chi-square statistic
    chi2 = ((AA - AA_exp)**2 / AA_exp +
            (Aa - Aa_exp)**2 / Aa_exp +
            (aa - aa_exp)**2 / aa_exp)

    # 5% significance threshold for 1 df
    in_hwe = chi2 < 3.841

    return {
        "p_allele": round(p, 4),
        "q_allele": round(q, 4),
        "chi2":     round(chi2, 4),
        "in_hwe":   in_hwe,
    }


# Test 1 — population in HWE (p=0.6, q=0.4)
res1 = hardy_weinberg_test(AA=360, Aa=480, aa=160)
print(f"Test 1: {res1}")
# Expected: p≈0.6, q≈0.4, chi2≈0.0, in_hwe=True

# Test 2 — population out of HWE
res2 = hardy_weinberg_test(AA=500, Aa=100, aa=400)
print(f"Test 2: {res2}")
# Expected: in_hwe=False

## Problem 3: CRISPR Guide RNA Efficiency Score (2 stars)

In [ ]:
def score_grna(guide: str) -> dict:
    """
    Score a 20-nt CRISPR guide RNA sequence for predicted efficiency.

    Args:
        guide: 20-character DNA string (A/T/G/C, upper-case).

    Returns:
        Dict with keys:
            'score'  : float, rounded to 4 dp
            'warning': str describing the issue, or None if no warning
    """
    score = 0.0

    # GC content
    gc_count = guide.count('G') + guide.count('C')
    gc = gc_count / 20.0
    if 0.4 <= gc <= 0.6:
        score += 0.2
    else:
        score -= 0.2

    # PAM-proximal nucleotide (position 20, 1-indexed = guide[19])
    pam_proximal = guide[19]
    if pam_proximal == 'G':
        score += 0.3
    elif pam_proximal == 'C':
        score += 0.1
    else:  # A or T
        score -= 0.1

    # Homopolymer runs of 4+
    i = 0
    while i < len(guide):
        j = i
        while j < len(guide) and guide[j] == guide[i]:
            j += 1
        run_length = j - i
        if run_length >= 4:
            score -= 0.15
        i = j

    # T at position 1 (0-indexed position 0)
    if guide[0] == 'T':
        score -= 0.1

    # Warning conditions
    warning = None
    # Check for >4 consecutive identical nucleotides
    i = 0
    has_long_homopolymer = False
    while i < len(guide):
        j = i
        while j < len(guide) and guide[j] == guide[i]:
            j += 1
        if j - i > 4:
            has_long_homopolymer = True
            break
        i = j

    if has_long_homopolymer:
        warning = "Homopolymer run > 4 nt detected"
    elif gc < 0.2:
        warning = "GC content too low (< 20%)"
    elif gc > 0.8:
        warning = "GC content too high (> 80%)"

    return {"score": round(score, 4), "warning": warning}


# Test
guides = [
    "ACGTACGTACGTACGTACGG",  # GC=50%, pos20=G, no homopolymer, pos1=A
    "TAAAACGTACGTACGTACGG",  # pos1=T penalty
    "AAAAACGTACGTACGTACGG",  # homopolymer AAAAA (run of 5)
    "GCGCGCGCGCGCGCGCGCGG",  # high GC (95%) → warning
]
for g in guides:
    r = score_grna(g)
    print(f"{g}  score={r['score']:.4f}  warning={r['warning']}")

## Problem 4: De Bruijn Graph Assembly (2 stars)

In [ ]:
def assemble_debruijn(kmers: list[str]) -> str | None:
    """
    Assemble a sequence from k-mers via a De Bruijn graph Eulerian path.

    Args:
        kmers: List of DNA strings all of the same length k.

    Returns:
        Assembled sequence string, or None if no Eulerian path exists.
    """
    # Build De Bruijn graph: prefix -> list of suffixes
    graph = defaultdict(list)
    in_degree  = defaultdict(int)
    out_degree = defaultdict(int)

    for kmer in kmers:
        prefix = kmer[:-1]
        suffix = kmer[1:]
        graph[prefix].append(suffix)
        out_degree[prefix] += 1
        in_degree[suffix]  += 1

    # Collect all nodes
    all_nodes = set(graph.keys()) | set(in_degree.keys())

    # Check for Eulerian path/circuit
    start_nodes = []
    end_nodes   = []
    for node in all_nodes:
        diff = out_degree[node] - in_degree[node]
        if diff == 1:
            start_nodes.append(node)
        elif diff == -1:
            end_nodes.append(node)
        elif diff != 0:
            return None  # Not Eulerian

    if len(start_nodes) == 0 and len(end_nodes) == 0:
        # Eulerian circuit — start from any node with edges
        start = next((n for n in all_nodes if out_degree[n] > 0), None)
    elif len(start_nodes) == 1 and len(end_nodes) == 1:
        start = start_nodes[0]
    else:
        return None

    if start is None:
        return None

    # Hierholzer's algorithm
    stack = [start]
    path  = []
    local_graph = {n: list(edges) for n, edges in graph.items()}

    while stack:
        v = stack[-1]
        if local_graph.get(v):
            u = local_graph[v].pop(0)
            stack.append(u)
        else:
            path.append(stack.pop())

    path.reverse()

    # Verify all edges were used
    total_edges = sum(len(edges) for edges in graph.values())
    if len(path) != total_edges + 1:
        return None

    # Reconstruct sequence
    sequence = path[0]
    for node in path[1:]:
        sequence += node[-1]

    return sequence


# Test 1 — assembles to ACGTACGT
kmers1 = ["ACG", "CGT", "GTA", "TAC", "ACG", "CGT"]
print(f"Assembly 1: {assemble_debruijn(kmers1)}")
# Expected: ACGTACGT

# Test 2 — linear path
kmers2 = ["ATG", "TGC", "GCA", "CAT", "ATC"]
print(f"Assembly 2: {assemble_debruijn(kmers2)}")

# Test 3 — no Eulerian path
kmers3 = ["AAA", "CCC", "GGG"]
print(f"Assembly 3 (no path): {assemble_debruijn(kmers3)}")
# Expected: None

## Problem 5: Protein Identification from Mass Spec Fragments (2 stars)

In [ ]:
RESIDUE_MASSES = {
    'G': 57.021, 'A': 71.037, 'V': 99.068, 'L': 113.084, 'I': 113.084,
    'P': 97.053, 'F': 147.068, 'W': 186.079, 'M': 131.040, 'S': 87.032,
    'T': 101.048, 'C': 103.009, 'Y': 163.063, 'H': 137.059, 'D': 115.027,
    'E': 129.043, 'N': 114.043, 'Q': 128.058, 'K': 128.095, 'R': 156.101,
}


def match_peptide_fragments(
    peptide: str,
    observed_masses: list[float],
    tolerance: float = 0.02,
) -> dict:
    """
    Match observed fragment masses against theoretical b-ions for a peptide.

    Args:
        peptide: Amino acid sequence string (single-letter codes, upper-case).
        observed_masses: List of observed fragment masses (Da).
        tolerance: Mass tolerance in Da for a match.

    Returns:
        Dict with 'matched_ions' (int), 'total_ions' (int),
        and 'coverage' (float, rounded to 4 dp).
    """
    # Compute b-ions: cumulative sum for positions 1..len-1
    b_ions = []
    cumulative = 0.0
    for i in range(len(peptide) - 1):
        cumulative += RESIDUE_MASSES[peptide[i]]
        b_ions.append(cumulative)

    total_ions = len(b_ions)

    # Count matches within tolerance
    matched = 0
    for b_mass in b_ions:
        for obs in observed_masses:
            if abs(obs - b_mass) <= tolerance:
                matched += 1
                break

    coverage = matched / total_ions if total_ions > 0 else 0.0

    return {
        "matched_ions": matched,
        "total_ions":   total_ions,
        "coverage":     round(coverage, 4),
    }


# Test — peptide ACDEFG, compute b-ions and check perfect match
peptide = "ACDEFG"
# b-ions: b1=A, b2=AC, b3=ACD, b4=ACDE, b5=ACDEF
theoretical_b = [
    round(71.037, 3),
    round(71.037 + 103.009, 3),
    round(71.037 + 103.009 + 115.027, 3),
    round(71.037 + 103.009 + 115.027 + 129.043, 3),
    round(71.037 + 103.009 + 115.027 + 129.043 + 147.068, 3),
]
print("Theoretical b-ions:", theoretical_b)

# Perfect match
r1 = match_peptide_fragments(peptide, theoretical_b, tolerance=0.02)
print(f"Perfect match: {r1}")
# Expected: matched_ions=5, total_ions=5, coverage=1.0

# Partial match
r2 = match_peptide_fragments(peptide, [71.037, 174.046, 999.0], tolerance=0.02)
print(f"Partial match: {r2}")

## Problem 6: Numerical ODE — Lotka-Volterra (3 stars)

In [ ]:
def lotka_volterra(
    alpha: float,
    beta: float,
    gamma: float,
    delta: float,
    prey0: float,
    pred0: float,
    T: float,
    dt: float,
) -> list[tuple[float, float, float]]:
    """
    Solve Lotka-Volterra equations using Euler's method.

    Args:
        alpha: Prey birth rate.
        beta:  Predation rate.
        gamma: Predator death rate.
        delta: Predator growth rate from predation.
        prey0: Initial prey population.
        pred0: Initial predator population.
        T:     Total simulation time.
        dt:    Time step size.

    Returns:
        List of (t, prey, pred) tuples, one per time step including t=0.
    """
    trajectory = []
    t    = 0.0
    prey = prey0
    pred = pred0

    while t <= T + dt * 1e-9:  # include endpoint
        trajectory.append((round(t, 10), prey, pred))
        # Euler update (simultaneous)
        prey_next = prey + dt * (alpha*prey - beta*prey*pred)
        pred_next = pred + dt * (delta*prey*pred - gamma*pred)
        prey = prey_next
        pred = pred_next
        t += dt

    return trajectory


# Test
trajectory = lotka_volterra(
    alpha=1.0, beta=0.1, gamma=1.5, delta=0.075,
    prey0=10.0, pred0=5.0,
    T=1.0, dt=0.1,
)
print(f"Steps: {len(trajectory)}")
print("First 3 steps:")
for step in trajectory[:3]:
    print(f"  t={step[0]:.2f}  prey={step[1]:.4f}  pred={step[2]:.4f}")
# Expected: 11 steps (t=0.0 to t=1.0 inclusive)

## Problem 7: N50 Genome Assembly Metric (1 star)

In [ ]:
def compute_n50(contig_lengths: list[int]) -> dict:
    """
    Compute N50, L50, total base pairs, and number of contigs.

    Args:
        contig_lengths: List of contig lengths (positive integers).

    Returns:
        Dict with keys 'N50', 'L50', 'total_bp', 'num_contigs'.
    """
    sorted_lengths = sorted(contig_lengths, reverse=True)
    total_bp = sum(sorted_lengths)
    half_total = total_bp / 2.0

    cumulative = 0
    for i, length in enumerate(sorted_lengths):
        cumulative += length
        if cumulative >= half_total:
            return {
                "N50":        length,
                "L50":        i + 1,
                "total_bp":   total_bp,
                "num_contigs": len(contig_lengths),
            }

    return {"N50": 0, "L50": 0, "total_bp": total_bp, "num_contigs": len(contig_lengths)}


# Test
lengths = [80, 60, 50, 40, 30, 20, 10]
result = compute_n50(lengths)
print(result)
# total_bp = 290, 50% = 145
# sorted desc: 80, 60, 50, 40, 30, 20, 10
# cumsum:       80, 140, 190 → N50=50 at count=3
# Expected: {'N50': 50, 'L50': 3, 'total_bp': 290, 'num_contigs': 7}

## Problem 8: Single-Cell Leiden Clustering (3 stars)

In [ ]:
def louvain_communities(
    adjacency: np.ndarray,
    gamma: float = 1.0,
) -> tuple[list[int], float]:
    """
    One-pass Louvain-style greedy modularity optimisation.

    Args:
        adjacency: Symmetric adjacency matrix, shape (n, n). No self-loops.
        gamma:     Resolution parameter (higher = more communities).

    Returns:
        Tuple of (communities, Q) where communities[i] is the community
        label for node i, and Q is the final modularity rounded to 6 dp.
    """
    n = adjacency.shape[0]
    m = adjacency.sum() / 2.0  # total edge weight
    k = adjacency.sum(axis=1)  # degree of each node

    # Each node starts in its own community
    communities = list(range(n))

    def compute_modularity(comms):
        Q = 0.0
        for i in range(n):
            for j in range(n):
                if comms[i] == comms[j]:
                    Q += adjacency[i, j] - gamma * k[i] * k[j] / (2 * m)
        return Q / (2 * m)

    # One-pass greedy: try moving each node to a neighbor's community
    improved = True
    while improved:
        improved = False
        for node in range(n):
            current_comm = communities[node]
            best_comm    = current_comm
            best_Q       = compute_modularity(communities)

            # Try each neighbor's community
            neighbor_comms = set()
            for nb in range(n):
                if adjacency[node, nb] > 0:
                    neighbor_comms.add(communities[nb])

            for cand_comm in neighbor_comms:
                if cand_comm == current_comm:
                    continue
                communities[node] = cand_comm
                new_Q = compute_modularity(communities)
                if new_Q > best_Q:
                    best_Q   = new_Q
                    best_comm = cand_comm

            if best_comm != current_comm:
                communities[node] = best_comm
                improved = True
            else:
                communities[node] = current_comm

    Q = round(compute_modularity(communities), 6)
    return communities, Q


# Test — two clear cliques connected by one edge
# Nodes 0-2: fully connected clique; Nodes 3-5: fully connected clique
# One bridge edge: 2—3
A = np.array([
    [0, 1, 1, 0, 0, 0],
    [1, 0, 1, 0, 0, 0],
    [1, 1, 0, 1, 0, 0],
    [0, 0, 1, 0, 1, 1],
    [0, 0, 0, 1, 0, 1],
    [0, 0, 0, 1, 1, 0],
], dtype=float)

communities, Q = louvain_communities(A, gamma=1.0)
print(f"Communities: {communities}")
print(f"Modularity Q: {Q}")
# Expected: nodes 0-2 in one community, nodes 3-5 in another; Q > 0